# 14 Ocurrence ML

This copy notebook models `P(observed / reported flooding complaint event)`.

The negative label means no observed 311 flood complaint during the matched event window. It is not proof of physical no-flooding.

Split rule:
- one classic stratified `75/15/10` train/validation/test split
- stratification prioritizes the target plus `mayoral_administration` when feasible
- model selection is a hyperparameter-grid sweep on train/validation; test is untouched final evaluation

Models are restricted to course-aligned supervised methods:
- Logistic Regression
- Decision Tree
- Random Forest
- Gradient Boosting
- optional XGBoost / LightGBM / CatBoost only if installed and already part of the workflow

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
SRC_DIR = ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

In [2]:
from project_name.modeling_stage import (
    OCCURRENCE_BIAS_DIAGNOSTICS_PATH,
    OCCURRENCE_CALIBRATION_PATH,
    OCCURRENCE_FEATURE_IMPORTANCE_PATH,
    OCCURRENCE_LEAKAGE_AUDIT_PATH,
    OCCURRENCE_PREDICTIONS_PATH,
    OCCURRENCE_RESULTS_PATH,
    compact_modeling_summary,
    feature_catalog,
    load_modeling_frame,
    run_occurrence_ml,
    summarize_feature_groups,
    unavailable_expected_features,
)

balanced = load_modeling_frame(view_name="strict_main", include_geometry=False)
print(f"balanced rows: {len(balanced):,}")
print(balanced["occurrence"].value_counts(dropna=False).to_string())
display(summarize_feature_groups(balanced))
display(unavailable_expected_features(balanced))
display(feature_catalog(balanced).head(40))

balanced rows: 69,738
occurrence
True     34870
False    34868


,feature_group,expected_features,available_features,mean_non_null_rate,missing_features
0,event_time,8,8,0.936914,0
1,extra,18,18,0.971959,0
2,governance,3,3,1.000000,0
3,hydrometeorology,6,6,0.987104,0
4,infrastructure,4,4,0.500000,0
5,network,9,9,0.444444,0
6,socioeconomic,4,4,0.971551,0
7,spatial_controls,5,5,1.000000,0
8,targets,4,4,0.873828,0
9,terrain_coastal,8,8,0.999993,0


,feature_group,feature,available,dtype,non_null_rate,n_unique


,feature_group,feature,available,dtype,non_null_rate,n_unique
0,event_time,day_of_week,True,Int64,1.000000,7
1,event_time,duration,True,float64,1.000000,7452
2,event_time,end,True,datetime64[us],0.495311,31625
3,event_time,hour,True,Int64,1.000000,24
4,event_time,month,True,Int64,1.000000,12
5,event_time,season,True,string,1.000000,4
6,event_time,start,True,datetime64[us],1.000000,32123
7,event_time,storm_event_id,True,string,1.000000,117
8,extra,catch_basin_nearest_ft,True,float64,1.000000,46215
9,extra,component_size,True,int64,1.000000,60


In [ ]:
results, predictions, feature_importance = run_occurrence_ml(balanced)

display(compact_modeling_summary(results, ["pr_auc", "recall", "balanced_accuracy"]).head(30))
display(results.sort_values(["pr_auc", "recall"], ascending=False, kind="stable").head(30))
display(feature_importance.head(30))

print(f"saved: {OCCURRENCE_RESULTS_PATH}")
print(f"saved: {OCCURRENCE_PREDICTIONS_PATH}")
print(f"saved: {OCCURRENCE_FEATURE_IMPORTANCE_PATH}")
print(f"saved: {OCCURRENCE_BIAS_DIAGNOSTICS_PATH}")
print(f"saved: {OCCURRENCE_CALIBRATION_PATH}")
print("incremental checkpoints: data/processed/modeling/diagnostics/modeling_stage/checkpoints/occurrence")

/home/map10194/miniconda3/envs/urban-flood-stress/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/home/map10194/miniconda3/envs/urban-flood-stress/lib/python3.11/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/map10194/miniconda3/envs/urban-flood-stress/lib/python3.11/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/map10194/miniconda3/envs/urban-floo

In [ ]:
results = pd.read_csv(OCCURRENCE_RESULTS_PATH)
predictions = pd.read_parquet(OCCURRENCE_PREDICTIONS_PATH)
feature_importance = pd.read_csv(OCCURRENCE_FEATURE_IMPORTANCE_PATH)
bias = pd.read_csv(OCCURRENCE_BIAS_DIAGNOSTICS_PATH)
calibration = pd.read_csv(OCCURRENCE_CALIBRATION_PATH)
leakage_audit = pd.read_csv(OCCURRENCE_LEAKAGE_AUDIT_PATH)

metric_cols = [
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "pr_auc",
    "false_positive_rate",
    "false_negative_rate",
    "brier_score",
    "train_pr_auc",
    "validation_pr_auc",
    "test_pr_auc",
    "cv_validation_primary_score",
    "train_vs_validation_gap",
    "validation_vs_test_gap",
    "possible_overfitting",
    "possible_test_degradation",
]
display(results[["model_name", "split_strategy", "status"] + [c for c in metric_cols if c in results.columns]].query("status == 'ok'"))
display(bias.head(40))
display(leakage_audit[leakage_audit["excluded_from_predictors"]].head(30))

## Archetype Diagnostics for False Negatives

Clusters were learned from observed positive flood events only, so they are not used as occurrence predictors.
They are still useful for asking which reported-event archetypes are missed by the classifier.

In [ ]:
CLUSTER_PATH = ROOT / "data" / "processed" / "modeling" / "clustering_event_archetypes.parquet"
ANOMALY_PATH = ROOT / "data" / "processed" / "modeling" / "anomaly_event_scores.parquet"

if CLUSTER_PATH.exists() and ANOMALY_PATH.exists():
    clusters = pd.read_parquet(CLUSTER_PATH)[
        ["event_id", "combined_cluster_id", "cluster_label"]
    ].copy()
    anomalies = pd.read_parquet(ANOMALY_PATH)[
        ["event_id", "anomaly_score", "anomaly_flag", "anomaly_method_agreement"]
    ].copy()
    occurrence_cluster_diag = (
        predictions.merge(clusters, on="event_id", how="left")
        .merge(anomalies, on="event_id", how="left")
    )
    occurrence_cluster_diag = occurrence_cluster_diag[occurrence_cluster_diag["set_name"].eq("test")].copy()
    occurrence_cluster_diag = occurrence_cluster_diag[occurrence_cluster_diag["y_true"].astype(bool)].copy()
    occurrence_cluster_diag["false_negative"] = ~occurrence_cluster_diag["y_pred"].astype(bool)
    false_negative_by_cluster = (
        occurrence_cluster_diag.groupby(["combined_cluster_id", "cluster_label"], dropna=False)
        .agg(
            n_positive_test_events=("event_id", "size"),
            false_negative_count=("false_negative", "sum"),
            false_negative_rate=("false_negative", "mean"),
            mean_anomaly_score=("anomaly_score", "mean"),
        )
        .reset_index()
        .sort_values("false_negative_rate", ascending=False, kind="stable")
    )
    false_negative_by_cluster.to_csv(
        ROOT / "data" / "processed" / "modeling" / "diagnostics_occurrence_by_cluster.csv",
        index=False,
    )
    display(false_negative_by_cluster)
else:
    print("Cluster/anomaly outputs are not available yet. Run 13_clustering-anomaly first.")

In [ ]:
best_row = (
    results.query("status == 'ok'")
    .sort_values(["pr_auc", "recall", "balanced_accuracy"], ascending=False, kind="stable")
    .iloc[0]
)
best_predictions = predictions[
    (predictions["model_name"] == best_row["model_name"])
    & (predictions["split_strategy"] == best_row["split_strategy"])
    & (predictions["set_name"] == "test")
]

fig, axes = plt.subplots(2, 2, figsize=(16, 11), constrained_layout=True)
plot_results = results.query("status == 'ok'").copy()
plot_results["label"] = plot_results["model_name"] + " | " + plot_results["split_strategy"]
axes[0, 0].barh(plot_results["label"], plot_results["pr_auc"], color="#2563EB")
axes[0, 0].set_title("PR-AUC by Model and Split")
axes[0, 0].set_xlabel("PR-AUC")

axes[0, 1].barh(plot_results["label"], plot_results["false_negative_rate"], color="#DC2626")
axes[0, 1].set_title("False Negative Rate by Model and Split")
axes[0, 1].set_xlabel("FNR")

cm = pd.crosstab(best_predictions["y_true"], best_predictions["y_pred"])
im = axes[1, 0].imshow(cm.to_numpy(), cmap="Blues")
axes[1, 0].set_title(f"Confusion Matrix: {best_row['model_name']} | {best_row['split_strategy']}")
axes[1, 0].set_xticks(range(len(cm.columns)))
axes[1, 0].set_xticklabels(cm.columns.astype(str))
axes[1, 0].set_yticks(range(len(cm.index)))
axes[1, 0].set_yticklabels(cm.index.astype(str))
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        axes[1, 0].text(j, i, int(cm.iloc[i, j]), ha="center", va="center", color="black")
plt.colorbar(im, ax=axes[1, 0], shrink=0.8)

top_features = feature_importance.head(20).sort_values("importance", kind="stable")
axes[1, 1].barh(top_features["feature"], top_features["importance"], color="#059669")
axes[1, 1].set_title("Top Feature Importances")
axes[1, 1].set_xlabel("importance")
plt.show()

In [ ]:
if not calibration.empty:
    best_calibration = calibration[
        (calibration["model_name"] == best_row["model_name"])
        & (calibration["split_strategy"] == best_row["split_strategy"])
        & (calibration["set_name"] == "test")
    ]
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.plot([0, 1], [0, 1], linestyle="--", color="#64748B")
    ax.plot(
        best_calibration["mean_predicted_probability"],
        best_calibration["observed_rate"],
        marker="o",
        color="#2563EB",
    )
    ax.set_title("Calibration Curve")
    ax.set_xlabel("mean predicted probability")
    ax.set_ylabel("observed event fraction")
    plt.show()
else:
    print("Calibration table is empty.")